In [1]:
import sys
import platform
from pathlib import Path

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"

print("Python:", sys.version.split()[0])
print("Architecture:", platform.machine())
print("Project exists:", PROJECT.exists())

for folder in ("src", "data", "reports", "tests"):
    print(folder, (PROJECT / folder).is_dir())

Python: 3.12.15
Architecture: s390x
Project exists: True
src True
data True
reports True
tests True


In [2]:
from pathlib import Path
import csv
import json
import hashlib
import platform
import sys

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"
DATA = PROJECT / "data"

print("=== Z-MIRROR WORKSPACE ===")
print("Python:", sys.version.split()[0])
print("Architecture:", platform.machine())
print("Project exists:", PROJECT.exists())

for folder in ("src", "data", "reports", "tests"):
    print(f"{folder}:", (PROJECT / folder).is_dir())

candidate = DATA / "generated_candidate.csv"
manifest_path = DATA / "generated_candidate.csv.manifest.json"

assert candidate.exists(), "Generated candidate CSV is missing"
assert manifest_path.exists(), "Generator manifest is missing"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

with candidate.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    row_count = sum(1 for _ in reader)

actual_hash = hashlib.sha256(candidate.read_bytes()).hexdigest()

print("\n=== GENERATOR VERIFICATION ===")
print("Columns:", columns)
print("Records:", row_count)
print("Seed:", manifest.get("seed"))
print("SHA-256 matches manifest:", actual_hash == manifest.get("sha256"))

assert platform.machine() == "s390x"
assert columns == ["record_id", "age", "balance", "outcome"]
assert row_count == 1000
assert actual_hash == manifest.get("sha256")

print("\nWORKSPACE VERIFICATION: PASS")


=== Z-MIRROR WORKSPACE ===
Python: 3.12.15
Architecture: s390x
Project exists: True
src: True
data: True
reports: True
tests: True

=== GENERATOR VERIFICATION ===
Columns: ['record_id', 'age', 'balance', 'outcome']
Records: 1000
Seed: 42
SHA-256 matches manifest: True

WORKSPACE VERIFICATION: PASS


In [3]:

from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from io import TextIOWrapper
import csv
import hashlib

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"
DATA = PROJECT / "data"
SOURCE_DIR = DATA / "source"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = SOURCE_DIR / "bank_marketing.zip"
REFERENCE_PATH = DATA / "reference_candidate.csv"
SOURCE_NOTE = SOURCE_DIR / "SOURCE_AND_ATTRIBUTION.txt"

URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

# Never silently replace an earlier reference file.
if REFERENCE_PATH.exists():
    raise FileExistsError(
        f"{REFERENCE_PATH} already exists. Inspect it before retrying."
    )

# Download the official dataset archive only if not already present.
if not ZIP_PATH.exists():
    print("Downloading official UCI Bank Marketing dataset...")
    urlretrieve(URL, ZIP_PATH)

zip_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("Downloaded archive SHA-256:", zip_hash)

# Find the bank-full.csv member in the official archive.
with ZipFile(ZIP_PATH) as zf:
    matches = [
        name for name in zf.namelist()
        if name.rstrip("/").split("/")[-1] == "bank-full.csv"
    ]
    if not matches:
        raise FileNotFoundError(
            "bank-full.csv was not found inside the downloaded archive."
        )

    member = matches[0]
    print("Reference file inside archive:", member)

    temp_path = REFERENCE_PATH.with_suffix(".csv.tmp")
    if temp_path.exists():
        raise FileExistsError(
            f"{temp_path} already exists. Inspect it before retrying."
        )

    rows_written = 0
    with zf.open(member) as raw, \
         TextIOWrapper(raw, encoding="utf-8-sig", newline="") as text, \
         temp_path.open("x", encoding="utf-8", newline="") as output:

        reader = csv.DictReader(text, delimiter=";")
        required = {"age", "balance", "y"}

        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(
                f"Unexpected source columns: {reader.fieldnames}"
            )

        writer = csv.DictWriter(
            output,
            fieldnames=["record_id", "age", "balance", "outcome"]
        )
        writer.writeheader()

        for index, row in enumerate(reader, start=1):
            if not row.get("age") or not row.get("balance") or not row.get("y"):
                continue

            writer.writerow({
                "record_id": f"UCI{index:06d}",
                "age": row["age"].strip(),
                "balance": row["balance"].strip(),
                "outcome": row["y"].strip().lower()
            })
            rows_written += 1

if rows_written == 0:
    temp_path.unlink(missing_ok=True)
    raise ValueError("No usable reference rows were produced.")

temp_path.replace(REFERENCE_PATH)

SOURCE_NOTE.write_text(
    "Dataset: Bank Marketing (bank-full.csv)\n"
    "Source: UCI Machine Learning Repository\n"
    "URL: https://archive.ics.uci.edu/dataset/222/bank+marketing\n"
    "DOI: 10.24432/C5K306\n"
    "Licence: Creative Commons Attribution 4.0 International (CC BY 4.0)\n"
    "Prepared fields: record_id, age, balance, outcome\n"
    "record_id is a synthetic row sequence, not a source identifier.\n"
    f"Source archive SHA-256: {zip_hash}\n",
    encoding="utf-8"
)

print("\n=== REFERENCE DATA PREPARED ===")
print("Rows:", rows_written)
print("File:", REFERENCE_PATH)
print("Columns: record_id, age, balance, outcome")
print("Attribution saved:", SOURCE_NOTE.exists())
print("Reference preparation: PASS")


Downloaded archive SHA-256: e0bf5f5de5b846e2f18e9d90606637267d46dfa260e0f17bb12e605db5efbeb4


FileNotFoundError: bank-full.csv was not found inside the downloaded archive.

In [4]:

from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from io import BytesIO, TextIOWrapper
import csv
import hashlib
import json

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"
DATA = PROJECT / "data"
SOURCE_DIR = DATA / "source"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = SOURCE_DIR / "bank_marketing.zip"
REFERENCE_PATH = DATA / "reference_candidate.csv"
SOURCE_NOTE = SOURCE_DIR / "SOURCE_AND_ATTRIBUTION.txt"
TEMP_PATH = DATA / "reference_candidate.csv.tmp"

URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

# Preserve any existing results.
if REFERENCE_PATH.exists() or TEMP_PATH.exists():
    raise FileExistsError(
        "Reference output or temporary file already exists. "
        "Inspect it before proceeding; nothing was overwritten."
    )

# Reuse the successfully downloaded archive.
if not ZIP_PATH.exists():
    print("Downloading official UCI archive...")
    urlretrieve(URL, ZIP_PATH)

archive_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()

# The outer ZIP contains bank.zip. The inner ZIP contains bank-full.csv.
with ZipFile(ZIP_PATH) as outer:
    bank_zips = [
        name for name in outer.namelist()
        if name.rstrip("/").split("/")[-1] == "bank.zip"
    ]
    if not bank_zips:
        raise FileNotFoundError(
            "bank.zip not found in the outer archive. "
            "Inspect the archive contents before retrying."
        )

    inner_bytes = outer.read(bank_zips[0])

with ZipFile(BytesIO(inner_bytes)) as inner:
    csv_files = [
        name for name in inner.namelist()
        if name.rstrip("/").split("/")[-1] == "bank-full.csv"
    ]
    if not csv_files:
        raise FileNotFoundError(
            "bank-full.csv not found inside bank.zip."
        )

    source_member = csv_files[0]
    rows_written = 0

    with inner.open(source_member) as raw, \
         TextIOWrapper(raw, encoding="utf-8-sig", newline="") as source, \
         TEMP_PATH.open("x", encoding="utf-8", newline="") as output:

        reader = csv.DictReader(source, delimiter=";")
        required = {"age", "balance", "y"}

        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(
                f"Unexpected columns in source: {reader.fieldnames}"
            )

        writer = csv.DictWriter(
            output,
            fieldnames=["record_id", "age", "balance", "outcome"]
        )
        writer.writeheader()

        for index, row in enumerate(reader, start=1):
            if not row.get("age") or not row.get("balance") or not row.get("y"):
                continue

            writer.writerow({
                "record_id": f"UCI{index:06d}",
                "age": row["age"].strip(),
                "balance": row["balance"].strip(),
                "outcome": row["y"].strip().lower()
            })
            rows_written += 1

if rows_written == 0:
    TEMP_PATH.unlink(missing_ok=True)
    raise ValueError("No usable reference rows were produced.")

TEMP_PATH.replace(REFERENCE_PATH)

SOURCE_NOTE.write_text(
    "Dataset: Bank Marketing (bank-full.csv)\n"
    "Source: UCI Machine Learning Repository\n"
    "URL: https://archive.ics.uci.edu/dataset/222/bank+marketing\n"
    "DOI: 10.24432/C5K306\n"
    "Licence: Creative Commons Attribution 4.0 International (CC BY 4.0)\n"
    "Prepared columns: record_id, age, balance, outcome\n"
    "record_id is a generated row sequence, not a source identifier.\n"
    f"Outer archive SHA-256: {archive_hash}\n"
    f"Source member: {source_member}\n",
    encoding="utf-8"
)

print("=== REFERENCE DATA PREPARED ===")
print("Source member:", source_member)
print("Rows prepared:", rows_written)
print("Reference CSV:", REFERENCE_PATH)
print("Attribution saved:", SOURCE_NOTE.exists())
print("Reference preparation: PASS")



=== REFERENCE DATA PREPARED ===
Source member: bank-full.csv
Rows prepared: 45211
Reference CSV: /home/ibm-user/shared/z-mirror-jupyter/data/reference_candidate.csv
Attribution saved: True
Reference preparation: PASS


In [5]:
 
from pathlib import Path
from collections import defaultdict
from bisect import bisect_right
from datetime import datetime, timezone
import csv
import statistics
import json
import hashlib

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"
DATA = PROJECT / "data"
REPORTS = PROJECT / "reports"

REFERENCE = DATA / "reference_candidate.csv"
SYNTHETIC = DATA / "generated_candidate.csv"

assert REFERENCE.exists(), f"Missing reference: {REFERENCE}"
assert SYNTHETIC.exists(), f"Missing synthetic data: {SYNTHETIC}"

def read_records(path):
    with path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    required = {"record_id", "age", "balance", "outcome"}
    if not rows or not required.issubset(rows[0].keys()):
        raise ValueError(f"Empty data or missing columns: {path}")

    result = []
    for line_number, row in enumerate(rows, start=2):
        try:
            age = int(row["age"])
            balance = int(row["balance"])
            outcome = row["outcome"].strip().lower()
            record_id = row["record_id"].strip()
        except (ValueError, TypeError, AttributeError) as exc:
            raise ValueError(
                f"Invalid values at line {line_number} in {path}"
            ) from exc

        if not record_id or outcome not in {"yes", "no"}:
            raise ValueError(f"Invalid ID/outcome at line {line_number}")
        result.append({
            "record_id": record_id,
            "age": age,
            "balance": balance,
            "outcome": outcome,
        })
    return result

def ks_distance(a, b):
    """Maximum empirical CDF difference; descriptive, not a privacy score."""
    if not a or not b:
        raise ValueError("Both datasets must contain records")
    sa, sb = sorted(a), sorted(b)
    values = sorted(set(sa) | set(sb))
    maximum = 0.0
    for x in values:
        cdf_a = bisect_right(sa, x) / len(sa)
        cdf_b = bisect_right(sb, x) / len(sb)
        maximum = max(maximum, abs(cdf_a - cdf_b))
    return maximum

def summary(values):
    return {
        "mean": statistics.mean(values),
        "median": statistics.median(values),
        "min": min(values),
        "max": max(values),
        "population_sd": statistics.pstdev(values),
    }

def yes_rate(rows):
    return sum(r["outcome"] == "yes" for r in rows) / len(rows)

def age_group(age):
    if age < 30:
        return "18-29"
    if age < 45:
        return "30-44"
    if age <= 60:
        return "45-60"
    return "61+"

def balance_group(balance):
    if balance < 0:
        return "negative"
    if balance == 0:
        return "zero"
    return "positive"

reference = read_records(REFERENCE)
synthetic = read_records(SYNTHETIC)

ref_age = [r["age"] for r in reference]
syn_age = [r["age"] for r in synthetic]
ref_balance = [r["balance"] for r in reference]
syn_balance = [r["balance"] for r in synthetic]

ref_age_stats = summary(ref_age)
syn_age_stats = summary(syn_age)
ref_bal_stats = summary(ref_balance)
syn_bal_stats = summary(syn_balance)

metrics = {
    "reference_rows": len(reference),
    "synthetic_rows": len(synthetic),
    "reference_sha256": hashlib.sha256(REFERENCE.read_bytes()).hexdigest(),
    "synthetic_sha256": hashlib.sha256(SYNTHETIC.read_bytes()).hexdigest(),
    "age": {
        "reference": ref_age_stats,
        "synthetic": syn_age_stats,
        "ks_distance": ks_distance(ref_age, syn_age),
    },
    "balance": {
        "reference": ref_bal_stats,
        "synthetic": syn_bal_stats,
        "ks_distance": ks_distance(ref_balance, syn_balance),
    },
    "outcome_yes_rate": {
        "reference": yes_rate(reference),
        "synthetic": yes_rate(synthetic),
    },
}

metrics["outcome_yes_rate"]["absolute_difference_pp"] = (
    abs(metrics["outcome_yes_rate"]["reference"]
        - metrics["outcome_yes_rate"]["synthetic"]) * 100
)

group_results = {}
for name, group_function in [
    ("age_group", age_group),
    ("balance_sign", balance_group),
]:
    group_results[name] = {}
    for label in sorted({group_function(r["age"] if name == "age_group"
                                        else r["balance"])
                         for r in reference + synthetic}):
        ref_group = [
            r for r in reference
            if group_function(r["age"] if name == "age_group"
                              else r["balance"]) == label
        ]
        syn_group = [
            r for r in synthetic
            if group_function(r["age"] if name == "age_group"
                              else r["balance"]) == label
        ]
        group_results[name][label] = {
            "reference_count": len(ref_group),
            "synthetic_count": len(syn_group),
            "reference_yes_rate": (
                yes_rate(ref_group) if ref_group else None
            ),
            "synthetic_yes_rate": (
                yes_rate(syn_group) if syn_group else None
            ),
        }
        ref_rate = group_results[name][label]["reference_yes_rate"]
        syn_rate = group_results[name][label]["synthetic_yes_rate"]
        group_results[name][label]["absolute_difference_pp"] = (
            abs(ref_rate - syn_rate) * 100
            if ref_rate is not None and syn_rate is not None
            else None
        )

metrics["conditional_outcome_rates"] = group_results
metrics["evaluation_notes"] = [
    "Generated data uses a hand-defined parameterized demonstration profile.",
    "KS distance is a descriptive distribution difference, not a privacy score.",
    "No utility acceptance thresholds have been agreed; no PASS/FAIL is assigned.",
    "These comparisons do not prove anonymity, general utility, or production fitness.",
]

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
report_path = REPORTS / f"utility_comparison_{timestamp}.json"
REPORTS.mkdir(parents=True, exist_ok=True)
report_path.write_text(
    json.dumps(metrics, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=== Z-MIRROR UTILITY COMPARISON ===")
print("Reference rows:", len(reference))
print("Synthetic rows:", len(synthetic))
print("Age KS distance:", round(metrics["age"]["ks_distance"], 4))
print("Balance KS distance:", round(metrics["balance"]["ks_distance"], 4))
print(
    "Reference outcome=yes rate:",
    round(metrics["outcome_yes_rate"]["reference"] * 100, 2), "%"
)
print(
    "Synthetic outcome=yes rate:",
    round(metrics["outcome_yes_rate"]["synthetic"] * 100, 2), "%"
)
print(
    "Absolute outcome-rate difference:",
    round(metrics["outcome_yes_rate"]["absolute_difference_pp"], 2),
    "percentage points"
)
print("\n=== CONDITIONAL OUTCOME RATES ===")
for dimension, groups in group_results.items():
    print(f"\n{dimension}:")
    for label, result in groups.items():
        print(
            label,
            "| n_ref =", result["reference_count"],
            "| n_syn =", result["synthetic_count"],
            "| yes_ref =", (
                None if result["reference_yes_rate"] is None
                else round(result["reference_yes_rate"] * 100, 2)
            ),
            "| yes_syn =", (
                None if result["synthetic_yes_rate"] is None
                else round(result["synthetic_yes_rate"] * 100, 2)
            ),
            "| difference_pp =", (
                None if result["absolute_difference_pp"] is None
                else round(result["absolute_difference_pp"], 2)
            ),
        )

print("\nSaved report:", report_path)
print("UTILITY EVALUATION: MEASURED; THRESHOLDS NOT YET DEFINED")



=== Z-MIRROR UTILITY COMPARISON ===
Reference rows: 45211
Synthetic rows: 1000
Age KS distance: 0.3361
Balance KS distance: 0.7511
Reference outcome=yes rate: 11.7 %
Synthetic outcome=yes rate: 43.9 %
Absolute outcome-rate difference: 32.2 percentage points

=== CONDITIONAL OUTCOME RATES ===

age_group:
18-29 | n_ref = 5273 | n_syn = 50 | yes_ref = 17.6 | yes_syn = 36.0 | difference_pp = 18.4
30-44 | n_ref = 24274 | n_syn = 283 | yes_ref = 10.12 | yes_syn = 42.76 | difference_pp = 32.64
45-60 | n_ref = 14476 | n_syn = 309 | yes_ref = 9.69 | yes_syn = 48.87 | difference_pp = 39.18
61+ | n_ref = 1188 | n_syn = 358 | yes_ref = 42.26 | yes_syn = 41.62 | difference_pp = 0.64

balance_sign:
negative | n_ref = 3766 | n_syn = 129 | yes_ref = 5.58 | yes_syn = 26.36 | difference_pp = 20.78
positive | n_ref = 37931 | n_syn = 871 | yes_ref = 12.62 | yes_syn = 46.5 | difference_pp = 33.88
zero | n_ref = 3514 | n_syn = 0 | yes_ref = 8.31 | yes_syn = None | difference_pp = None

Saved report: /home/i

In [6]:

from pathlib import Path
from collections import Counter, defaultdict
from bisect import bisect_right
from datetime import datetime, timezone
import csv
import random
import json
import hashlib

PROJECT = Path.home() / "shared" / "z-mirror-jupyter"
DATA = PROJECT / "data"

REFERENCE = DATA / "reference_candidate.csv"
OUTPUT = DATA / "generated_candidate_v2.csv"
MANIFEST = DATA / "generated_candidate_v2.csv.manifest.json"

SEED = 42
N_ROWS = 1000
N_BALANCE_BINS = 8

# Never overwrite previous results.
if OUTPUT.exists() or MANIFEST.exists():
    raise FileExistsError(
        "Generator v2 output already exists. Inspect it before retrying."
    )

# Load and validate the permitted, public reference data.
reference = []
with REFERENCE.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    required = {"record_id", "age", "balance", "outcome"}
    if not reader.fieldnames or not required.issubset(reader.fieldnames):
        raise ValueError(f"Unexpected reference columns: {reader.fieldnames}")

    for line, row in enumerate(reader, start=2):
        try:
            age = int(row["age"])
            balance = int(row["balance"])
            outcome = row["outcome"].strip().lower()
        except (ValueError, TypeError, AttributeError) as exc:
            raise ValueError(f"Invalid reference row {line}") from exc

        if not 18 <= age <= 100:
            raise ValueError(f"Age outside supported range at row {line}")
        if not -9_999_999 <= balance <= 9_999_999:
            raise ValueError(f"Balance outside supported range at row {line}")
        if outcome not in {"yes", "no"}:
            raise ValueError(f"Invalid outcome at row {line}")

        reference.append((age, balance, outcome))

if not reference:
    raise ValueError("Reference dataset is empty")

# Create coarse age groups and separate negative, zero and positive balances.
def age_group(age):
    if age < 30:
        return "18-29"
    if age < 45:
        return "30-44"
    if age <= 60:
        return "45-60"
    return "61+"

negative = sorted(b for _, b, _ in reference if b < 0)
positive = sorted(b for _, b, _ in reference if b > 0)

def quantile_cuts(values, bins=N_BALANCE_BINS):
    if not values:
        return []
    cuts = []
    for i in range(1, min(bins, len(values))):
        index = min(len(values) - 1, i * len(values) // bins)
        cuts.append(values[index])
    return sorted(set(cuts))

negative_cuts = quantile_cuts(negative)
positive_cuts = quantile_cuts(positive)

def balance_bucket(balance):
    if balance == 0:
        return "zero"
    if balance < 0:
        return f"negative_{bisect_right(negative_cuts, balance)}"
    return f"positive_{bisect_right(positive_cuts, balance)}"

# Fit only aggregate counts and bucket ranges; do not retain whole source rows.
cells = defaultdict(lambda: {"count": 0, "age_counts": Counter()})
bucket_bounds = {}

for age, balance, outcome in reference:
    bucket = balance_bucket(balance)
    key = (age_group(age), bucket, outcome)

    cells[key]["count"] += 1
    cells[key]["age_counts"][age] += 1

    if bucket not in bucket_bounds:
        bucket_bounds[bucket] = [balance, balance]
    else:
        bucket_bounds[bucket][0] = min(bucket_bounds[bucket][0], balance)
        bucket_bounds[bucket][1] = max(bucket_bounds[bucket][1], balance)

keys = list(cells)
weights = [cells[key]["count"] for key in keys]
rng = random.Random(SEED)

# Generate from weighted aggregate cells, then sample age and balance within
# the selected coarse cell/bucket. No complete source record is copied.
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
temp_output = OUTPUT.with_suffix(".csv.tmp")
temp_manifest = Path(str(MANIFEST) + ".tmp")

if temp_output.exists() or temp_manifest.exists():
    raise FileExistsError("Temporary v2 file exists; inspect it before retrying")

with temp_output.open("x", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["record_id", "age", "balance", "outcome"])

    for i in range(1, N_ROWS + 1):
        age_band, bucket, outcome = rng.choices(keys, weights=weights, k=1)[0]

        age_counts = cells[(age_band, bucket, outcome)]["age_counts"]
        ages = list(age_counts)
        age = rng.choices(
            ages, weights=[age_counts[a] for a in ages], k=1
        )[0]

        low, high = bucket_bounds[bucket]
        balance = rng.randint(low, high)

        writer.writerow((f"SYN2_{i:06d}", age, balance, outcome))

temp_output.replace(OUTPUT)

output_hash = hashlib.sha256(OUTPUT.read_bytes()).hexdigest()
reference_hash = hashlib.sha256(REFERENCE.read_bytes()).hexdigest()

manifest = {
    "generator": "zmirror-aggregate-binned-v2",
    "seed": SEED,
    "rows": N_ROWS,
    "schema": ["record_id", "age", "balance", "outcome"],
    "reference_dataset": "UCI Bank Marketing, prepared subset",
    "reference_sha256": reference_hash,
    "generated_sha256": output_hash,
    "age_groups": ["18-29", "30-44", "45-60", "61+"],
    "negative_balance_bins": len(negative_cuts) + 1,
    "positive_balance_bins": len(positive_cuts) + 1,
    "aggregate_cells": len(cells),
    "method": (
        "Weighted sampling from aggregate age-group, balance-bin and outcome "
        "counts; age sampled from within-cell counts; balance sampled uniformly "
        "within the observed bucket range."
    ),
    "limitations": [
        "This is a coarse aggregate-based demonstration, not a formal privacy mechanism.",
        "No differential privacy guarantee is provided.",
        "Statistical utility must be measured separately.",
        "Small aggregate cells and generated values may still carry disclosure risk."
    ]
}

with temp_manifest.open("x", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
    f.write("\n")

temp_manifest.replace(MANIFEST)

print("=== GENERATOR V2 ===")
print("Reference rows:", len(reference))
print("Generated rows:", N_ROWS)
print("Aggregate cells:", len(cells))
print("Seed:", SEED)
print("Output:", OUTPUT)
print("Manifest:", MANIFEST)
print("SHA-256:", output_hash)
print("Utility and privacy guarantees: NOT ESTABLISHED")
print("GENERATOR V2: PASS")


=== GENERATOR V2 ===
Reference rows: 45211
Generated rows: 1000
Aggregate cells: 124
Seed: 42
Output: /home/ibm-user/shared/z-mirror-jupyter/data/generated_candidate_v2.csv
Manifest: /home/ibm-user/shared/z-mirror-jupyter/data/generated_candidate_v2.csv.manifest.json
SHA-256: 9c85fd9e7edffef1ebdb5898207d29906d508672b8a9ea349f1f56597c5f42ce
Utility and privacy guarantees: NOT ESTABLISHED
GENERATOR V2: PASS


In [7]:
SYNTHETIC = DATA / "generated_candidate_v2.csv"

In [8]:
SYNTHETIC = DATA / "generated_candidate_v2.csv"

In [9]:
SYNTHETIC = DATA / "generated_candidate_v2.csv"
